In [ ]:
import pandas as pd
import json
import os

In [ ]:
# Load Data
df = pd.read_csv('/content/Data_Kemiskinan_Sumbar.csv', index_col=0)
df.index = df.index.str.strip()

TAHUN_LIST = [str(t) for t in range(2005, 2026)]

# Mapping nama CSV → nama file GeoJSON
MAPPING_NAMA = {
    'Kepulauan Mentawai': 'Kepulauan Mentawai',
    'Pesisir Selatan':    'Pesisir Selatan',
    'Kab.Solok':          'Solok',
    'Sijunjung':          'Sijunjung',
    'Tanah Datar':        'Tanah Datar',
    'Padang Pariaman':    'Padang Pariaman',
    'Agam':               'Agam',
    'Lima Puluh Kota':    'Lima Puluh Kota',
    'Pasaman':            'Pasaman',
    'Solok Selatan':      'Solok Selatan',
    'Dharmasraya':        'Dharmasraya',
    'Pasaman Barat':      'Pasaman Barat',
    'Padang':             'Padang',
    'Kota Solok':         'Kota Solok',
    'Sawahlunto':         'Sawahlunto',
    'Padang Panjang':     'Padang Panjang',
    'Bukittinggi':        'Bukittinggi',
    'Payakumbuh':         'Payakumbuh',
    'Pariaman':           'Pariaman',
}

KOTA_LIST = ['Bukittinggi', 'Padang', 'Padang Panjang', 'Pariaman',
             'Payakumbuh', 'Sawahlunto', 'Kota Solok']

data_kemiskinan = {}
for nama_csv, nama_geo in MAPPING_NAMA.items():
    if nama_csv in df.index:
        data_kemiskinan[nama_geo] = {t: int(df.loc[nama_csv, t]) for t in TAHUN_LIST}

In [ ]:
# Dropdown Option
dropdown_options = ''
for kab in sorted(data_kemiskinan.keys()):
    if kab in KOTA_LIST:
        display_name = f"Kota {kab}"
    elif kab == 'Kepulauan Mentawai':
        display_name = kab
    else:
        display_name = f"Kab. {kab}"
    dropdown_options += f'<option value="{kab}">{display_name}</option>\n'

In [ ]:
# Load GeoJson
with open('/content/Kabupaten.geojson', 'r') as f:
    geojson_kab = json.load(f)

geojson_kecamatan = {}
for nama_geo in MAPPING_NAMA.values():
    path = f'/content/Kabupaten/{nama_geo}.geojson'
    if os.path.exists(path):
        with open(path, 'r') as f:
            geojson_kecamatan[nama_geo] = json.load(f)

In [ ]:
# HTML
data_js        = json.dumps(data_kemiskinan, ensure_ascii=False)
geojson_kab_js = json.dumps(geojson_kab, ensure_ascii=False)
geojson_kec_js = json.dumps(geojson_kecamatan, ensure_ascii=False)

html_content = f"""<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Garis Kemiskinan Sumatera Barat 2005–2025</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800&display=swap" rel="stylesheet">

    <style>
        :root {{
            --sidebar-bg:    #ffffff;
            --bg-section:    #f8f9fa;
            --accent-blue:   #3498db;
            --accent-green:  #2ecc71;
            --accent-yellow: #f1c40f;
            --accent-orange: #e67e22;
            --accent-red:    #c0392b;
            --text-main:     #2c3e50;
            --text-sub:      #7f8c8d;
            --text-dim:      #aab0b5;
            --border:        #e8ecef;
            --shadow:        rgba(0,0,0,0.08);
            --font:          'Nunito', 'Segoe UI', sans-serif;
        }}

        * {{ margin: 0; padding: 0; box-sizing: border-box; }}

        body {{
            font-family: var(--font);
            background: #eef2f5;
            display: flex;
            height: 100vh;
            overflow: hidden;
        }}

        /* ══ SIDEBAR ══ */
        .custom-sidebar {{
            width: 280px;
            min-width: 280px;
            background: var(--sidebar-bg);
            box-shadow: 2px 0 10px var(--shadow);
            z-index: 100;
            overflow-y: auto;
            display: flex;
            flex-direction: column;
        }}

        .title {{
            background: var(--accent-blue);
            color: #fff;
            padding: 18px 20px 14px;
            text-align: center;
        }}

        .title h2 {{
            font-size: 13px;
            font-weight: 800;
            letter-spacing: 0.4px;
            line-height: 1.35;
            margin-bottom: 5px;
        }}

        .title p {{
            font-size: 11px;
            opacity: 0.88;
            margin-top: 2px;
        }}

        .year-display {{
            background: var(--bg-section);
            text-align: center;
            padding: 14px 20px 10px;
            border-bottom: 1px solid var(--border);
        }}

        .year-display h1 {{
            font-size: 42px;
            font-weight: 800;
            color: var(--accent-blue);
            line-height: 1;
        }}

        .year-display p {{
            font-size: 10px;
            color: var(--text-sub);
            margin-top: 4px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}

        .slider-container {{
            padding: 10px 20px 4px;
            border-bottom: 1px solid var(--border);
        }}

        input[type=range] {{
            -webkit-appearance: none;
            width: 100%;
            height: 5px;
            background: linear-gradient(to right,
                var(--accent-blue) var(--pct, 100%),
                var(--border) var(--pct, 100%));
            border-radius: 3px;
            outline: none;
            cursor: pointer;
            margin-bottom: 4px;
        }}

        input[type=range]::-webkit-slider-thumb {{
            -webkit-appearance: none;
            width: 16px;
            height: 16px;
            border-radius: 50%;
            background: var(--accent-blue);
            border: 2px solid #fff;
            box-shadow: 0 1px 4px rgba(52,152,219,0.5);
            cursor: pointer;
            transition: transform .15s;
        }}

        input[type=range]::-webkit-slider-thumb:hover {{ transform: scale(1.2); }}

        .playback-controls {{
            display: flex;
            gap: 6px;
            padding: 10px 20px;
            border-bottom: 1px solid var(--border);
        }}

        .playback-controls button {{
            flex: 1;
            padding: 7px 4px;
            border: 1px solid var(--border);
            border-radius: 5px;
            font-family: var(--font);
            font-size: 11px;
            font-weight: 700;
            cursor: pointer;
            transition: all .18s;
            background: #fff;
            color: var(--text-main);
        }}

        #playBtn  {{ background: var(--accent-blue);  color: #fff; border-color: var(--accent-blue); }}
        #pauseBtn {{ background: var(--accent-orange); color: #fff; border-color: var(--accent-orange); }}
        #resetBtn {{ background: #fff; color: var(--text-sub); }}

        .playback-controls button:hover:not(:disabled) {{
            filter: brightness(0.9);
            transform: translateY(-1px);
        }}

        .playback-controls button:disabled {{
            opacity: 0.4;
            cursor: not-allowed;
        }}

        .stats-panel {{
            padding: 10px 20px;
            border-bottom: 1px solid var(--border);
            background: var(--bg-section);
            font-size: 12px;
        }}

        .stats-panel p {{
            margin: 5px 0;
            display: flex;
            justify-content: space-between;
            align-items: baseline;
            flex-wrap: wrap;
            gap: 2px;
        }}

        .stat-label {{
            color: var(--text-sub);
            font-weight: 600;
            font-size: 11px;
            white-space: nowrap;
        }}

        .stat-value {{
            color: var(--text-main);
            font-weight: 700;
            font-size: 11px;
            text-align: right;
        }}

        .legend {{
            padding: 12px 20px;
            border-bottom: 1px solid var(--border);
        }}

        .legend h4 {{
            font-size: 11px;
            font-weight: 800;
            color: var(--text-main);
            margin-bottom: 10px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}

        .legend-bar {{
            height: 14px;
            border-radius: 7px;
            background: linear-gradient(to right,
                #3498db 0%,
                #5dade2 16%,
                #a8d8ea 28%,
                #f9e79f 40%,
                #f7c948 52%,
                #f0a500 62%,
                #e67e22 72%,
                #d35400 83%,
                #c0392b 100%);
            margin-bottom: 6px;
            box-shadow: inset 0 1px 3px rgba(0,0,0,0.1);
        }}

        .legend-bar-labels {{
            display: flex;
            justify-content: space-between;
            font-size: 9px;
            color: var(--text-dim);
            margin-bottom: 10px;
        }}

        .legend-item {{
            display: flex;
            align-items: center;
            margin-bottom: 6px;
            font-size: 11px;
            color: var(--text-main);
        }}

        .legend-color {{
            width: 22px;
            height: 13px;
            border-radius: 3px;
            margin-right: 8px;
            flex-shrink: 0;
            border: 1px solid rgba(0,0,0,0.08);
        }}

        .widget {{
            padding: 10px 20px;
            border-bottom: 1px solid var(--border);
        }}

        .widget button {{
            width: 100%;
            padding: 9px;
            background: #fff;
            color: var(--text-main);
            border: 1px solid var(--border);
            border-radius: 5px;
            font-family: var(--font);
            font-size: 12px;
            font-weight: 700;
            cursor: pointer;
            transition: all .18s;
        }}

        .widget button:hover {{
            background: var(--bg-section);
            border-color: var(--accent-blue);
            color: var(--accent-blue);
        }}

        select {{
            width: 100%;
            padding: 9px 10px;
            border: 1px solid var(--border);
            border-radius: 5px;
            font-family: var(--font);
            font-size: 12px;
            color: var(--text-main);
            background: #fff;
            cursor: pointer;
            outline: none;
        }}

        select:focus {{ border-color: var(--accent-blue); }}

        .info-text {{
            padding: 10px 20px 16px;
            font-size: 10px;
            color: var(--text-dim);
            line-height: 1.6;
            margin-top: auto;
        }}

        .custom-sidebar::-webkit-scrollbar {{ width: 4px; }}
        .custom-sidebar::-webkit-scrollbar-track {{ background: transparent; }}
        .custom-sidebar::-webkit-scrollbar-thumb {{ background: var(--border); border-radius: 2px; }}

        /* ══ MAP ══ */
        .map-container {{ flex: 1; position: relative; }}
        #map {{ height: 100%; width: 100%; }}

        .map-overlay {{
            position: absolute;
            top: 14px;
            right: 14px;
            background: rgba(255,255,255,0.97);
            backdrop-filter: blur(8px);
            border: 1px solid var(--border);
            border-radius: 10px;
            padding: 14px 18px;
            min-width: 210px;
            z-index: 500;
            display: none;
            box-shadow: 0 4px 16px var(--shadow);
        }}

        .overlay-label {{
            font-size: 9px;
            text-transform: uppercase;
            letter-spacing: 1px;
            color: var(--text-dim);
            font-weight: 700;
        }}

        .overlay-name {{
            font-size: 16px;
            font-weight: 800;
            color: var(--text-main);
            margin: 2px 0 8px;
        }}

        .overlay-val {{
            font-size: 22px;
            font-weight: 800;
            color: var(--accent-blue);
            line-height: 1;
        }}

        .overlay-kat {{
            font-size: 11px;
            color: var(--text-sub);
            margin-top: 2px;
            margin-bottom: 8px;
        }}

        .overlay-trend {{
            display: flex;
            gap: 3px;
            flex-wrap: wrap;
        }}

        .trend-dot {{
            width: 9px;
            height: 9px;
            border-radius: 50%;
            cursor: default;
        }}

        .leaflet-tooltip {{
            background: rgba(255,255,255,0.97) !important;
            border: 1px solid var(--border) !important;
            color: var(--text-main) !important;
            font-family: var(--font) !important;
            font-size: 12px !important;
            border-radius: 6px !important;
            box-shadow: 0 3px 10px var(--shadow) !important;
            padding: 8px 12px !important;
        }}

        .leaflet-tooltip::before {{
            border-top-color: var(--border) !important;
        }}
    </style>
</head>
<body>

<!-- ═══════════════ SIDEBAR ═══════════════ -->
<div class="custom-sidebar">

    <div class="title">
        <h2>📊 GARIS KEMISKINAN SUMATERA BARAT</h2>
        <p>Visualisasi Spasio-Temporal | 2005 – 2025</p>
        <p style="font-size: 10px; margin-top:3px;">Sumber: BPS Sumatera Barat</p>
    </div>

    <div class="year-display">
        <h1 id="tahunDisplay">2025</h1>
        <p>Garis Kemiskinan (Rp/kapita/bulan)</p>
    </div>

    <div class="slider-container">
        <input type="range" id="yearSlider" min="2005" max="2025"
               value="2025" step="1" oninput="onSliderInput(this.value)">
    </div>

    <div class="playback-controls">
        <button id="playBtn"  onclick="doPlay()">▶ Putar</button>
        <button id="pauseBtn" onclick="doPause()" disabled>⏸ Jeda</button>
        <button id="resetBtn" onclick="doReset()">⟳ Reset</button>
    </div>

    <div class="stats-panel" id="statsPanel">
        <p>
            <span class="stat-label">🏙️ Tertinggi:</span>
            <span>
                <span id="tertinggiWilayah" class="stat-value">-</span><br>
                <span id="tertinggiValue"   class="stat-value" style="color:var(--accent-red)">-</span>
            </span>
        </p>
        <p>
            <span class="stat-label">🏞️ Terendah:</span>
            <span>
                <span id="terendahWilayah" class="stat-value">-</span><br>
                <span id="terendahValue"   class="stat-value" style="color:var(--accent-blue)">-</span>
            </span>
        </p>
        <p>
            <span class="stat-label">📈 Rata-rata:</span>
            <span id="rataProvinsi" class="stat-value">-</span>
        </p>
    </div>

    <div class="legend">
        <h4>Klasifikasi Garis Kemiskinan</h4>
        <div class="legend-bar"></div>
        <div class="legend-bar-labels">
            <span>&lt;300rb</span>
            <span>400rb</span>
            <span>500rb</span>
            <span>&gt;600rb</span>
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background:#c0392b;"></div>
            <span>&gt; Rp 600.000 <em style="color:var(--text-dim)"> (Sangat Tinggi)</em></span>
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background:#e67e22;"></div>
            <span>Rp 500.000 – 600.000 <em style="color:var(--text-dim)"> (Tinggi)</em></span>
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background:#f1c40f;"></div>
            <span>Rp 400.000 – 500.000 <em style="color:var(--text-dim)"> (Sedang)</em></span>
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background:#2ecc71;"></div>
            <span>Rp 300.000 – 400.000 <em style="color:var(--text-dim)"> (Rendah)</em></span>
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background:#3498db;"></div>
            <span>&lt; Rp 300.000 <em style="color:var(--text-dim)"> (Sangat Rendah)</em></span>
        </div>
    </div>

    <div class="widget">
        <button id="resetViewBtn" onclick="resetView()">🗺️ Reset Peta</button>
    </div>

    <div class="widget">
        <select id="pilihKabupaten" onchange="zoomKabupaten(this.value)">
            <option value="">📌 Pilih Kabupaten/Kota untuk Detail</option>
            {dropdown_options}
        </select>
    </div>

    <div class="info-text">
        💡 Geser slider tahun untuk melihat perubahan<br>
        Hover/klik peta untuk melihat detail wilayah
    </div>

</div>

<!-- ═══════════════ MAP ═══════════════ -->
<div class="map-container">
    <div id="map"></div>

    <div class="map-overlay" id="mapOverlay">
        <div class="overlay-label">Kabupaten / Kota</div>
        <div class="overlay-name" id="ovName">—</div>
        <div class="overlay-label">Garis Kemiskinan</div>
        <div class="overlay-val"  id="ovVal">—</div>
        <div class="overlay-kat"  id="ovKat">—</div>
        <div class="overlay-label" style="margin-bottom:5px;">Tren 2005–2025</div>
        <div class="overlay-trend" id="ovTrend"></div>
    </div>
</div>

<!-- ═══════════════ SCRIPT ═══════════════ -->
<script>
const TAHUN_LIST      = {json.dumps(TAHUN_LIST)};
const dataKemiskinan  = {data_js};
const geojsonKabupaten = {geojson_kab_js};
const geojsonKecamatan = {geojson_kec_js};

let currentTahun = '2025';
let currentLayer = null;
let playTimer    = null;

/* ── MAP (light basemap) ── */
const map = L.map('map', {{ zoomControl: true, attributionControl: true }})
              .setView([-0.78, 100.5], 8);

L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
    attribution: '&copy; OpenStreetMap &copy; CartoDB',
    subdomains: 'abcd'
}}).addTo(map);

/* ══════════════════════════════════════════════
   WARNA — skala absolut Rp dengan interpolasi
   mulus antar 6 anchor (banyak gradasi warna)
══════════════════════════════════════════════ */
function nilaiKeWarna(nilai) {{
    if (!nilai || nilai === 0) return '#bdc3c7';

    // Anchor: [threshold_Rp, [R, G, B]]
    const anchors = [
        [0,       [52,  152, 219]],   // biru       < 300k
        [300000,  [46,  204, 113]],   // hijau      300-400k
        [400000,  [241, 196,  15]],   // kuning     400-500k
        [500000,  [230, 126,  34]],   // oranye     500-600k
        [600000,  [192,  57,  43]],   // merah      > 600k
        [900000,  [110,   5,   5]]    // merah tua  ujung atas
    ];

    if (nilai <= anchors[0][0])             return rgbHex(...anchors[0][1]);
    if (nilai >= anchors[anchors.length-1][0]) return rgbHex(...anchors[anchors.length-1][1]);

    for (let i = 0; i < anchors.length - 1; i++) {{
        const [t0, c0] = anchors[i];
        const [t1, c1] = anchors[i + 1];
        if (nilai >= t0 && nilai <= t1) {{
            const t = (nilai - t0) / (t1 - t0);
            return rgbHex(
                Math.round(c0[0] + (c1[0] - c0[0]) * t),
                Math.round(c0[1] + (c1[1] - c0[1]) * t),
                Math.round(c0[2] + (c1[2] - c0[2]) * t)
            );
        }}
    }}
    return '#bdc3c7';
}}

function rgbHex(r, g, b) {{
    const h = v => Math.max(0, Math.min(255, v)).toString(16).padStart(2, '0');
    return '#' + h(r) + h(g) + h(b);
}}

function kategori(nilai) {{
    if (!nilai) return '—';
    if (nilai > 600000) return 'Sangat Tinggi';
    if (nilai > 500000) return 'Tinggi';
    if (nilai > 400000) return 'Sedang';
    if (nilai > 300000) return 'Rendah';
    return 'Sangat Rendah';
}}

function formatRupiah(nilai) {{
    if (!nilai || nilai === 0) return '-';
    return 'Rp ' + nilai.toLocaleString('id-ID');
}}

/* ── NAME UTILS ── */
const NAME_MAP = {{
    'KEPULAUAN MENTAWAI':'Kepulauan Mentawai','PESISIR SELATAN':'Pesisir Selatan',
    'SOLOK':'Solok','SIJUNJUNG':'Sijunjung','TANAH DATAR':'Tanah Datar',
    'PADANG PARIAMAN':'Padang Pariaman','AGAM':'Agam','LIMA PULUH KOTA':'Lima Puluh Kota',
    'PASAMAN':'Pasaman','SOLOK SELATAN':'Solok Selatan','DHARMASRAYA':'Dharmasraya',
    'PASAMAN BARAT':'Pasaman Barat','PADANG':'Padang','KOTA SOLOK':'Kota Solok',
    'SAWAHLUNTO':'Sawahlunto','PADANG PANJANG':'Padang Panjang',
    'BUKITTINGGI':'Bukittinggi','PAYAKUMBUH':'Payakumbuh','PARIAMAN':'Pariaman'
}};

const KOTA = ['Bukittinggi','Padang','Padang Panjang','Pariaman',
              'Payakumbuh','Sawahlunto','Kota Solok'];

function normalisasiNama(nama) {{
    return NAME_MAP[nama.toUpperCase()] || nama;
}}

function getNamaFeature(feature) {{
    return feature.properties?.name    ||
           feature.properties?.NAMA    ||
           feature.properties?.NAMOBJ  ||
           feature.properties?.KABUPATEN ||
           feature.properties?.kabupaten ||
           feature.properties?.NAME_2  || '';
}}

function displayNama(namaStd) {{
    if (KOTA.includes(namaStd)) return 'Kota ' + namaStd;
    if (namaStd === 'Kepulauan Mentawai') return namaStd;
    return 'Kab. ' + namaStd;
}}

/* ── STYLE ── */
function styleFeature(feature) {{
    const namaStd = normalisasiNama(getNamaFeature(feature));
    const nilai   = (dataKemiskinan[namaStd] || {{}})[currentTahun];
    return {{
        fillColor:   nilaiKeWarna(nilai),
        weight:      1.2,
        opacity:     1,
        color:       '#ffffff',
        fillOpacity: nilai ? 0.80 : 0.25
    }};
}}

/* ── TOOLTIP & EVENTS ── */
function onEachFeature(feature, layer) {{
    const namaStd = normalisasiNama(getNamaFeature(feature));
    const nilai   = (dataKemiskinan[namaStd] || {{}})[currentTahun];
    const dispN   = displayNama(namaStd);

    const ttHtml = nilai
        ? `<strong>${{dispN}}</strong><br>
           Garis Kemiskinan: <strong>${{formatRupiah(nilai)}}</strong><br>
           Kategori: <em>${{kategori(nilai)}}</em>`
        : `<strong>${{namaStd}}</strong><br><em>Data tidak tersedia</em>`;

    layer.bindTooltip(ttHtml, {{ sticky: true, direction: 'top', opacity: 1 }});

    layer.on('mouseover', function() {{
        layer.setStyle({{ weight: 2.5, color: '#2c3e50', fillOpacity: 0.95 }});
        document.getElementById('mapOverlay').style.display = 'block';
        document.getElementById('ovName').textContent = dispN;
        document.getElementById('ovVal').textContent  = nilai ? formatRupiah(nilai) : 'N/A';
        document.getElementById('ovVal').style.color  = nilai ? nilaiKeWarna(nilai) : '#bdc3c7';
        document.getElementById('ovKat').textContent  = kategori(nilai);

        const trendEl = document.getElementById('ovTrend');
        trendEl.innerHTML = '';
        TAHUN_LIST.forEach(t => {{
            const v = (dataKemiskinan[namaStd] || {{}})[t];
            const dot = document.createElement('div');
            dot.className = 'trend-dot';
            dot.style.background = nilaiKeWarna(v);
            dot.title = t + ': ' + formatRupiah(v);
            trendEl.appendChild(dot);
        }});
    }});

    layer.on('mouseout', function() {{
        if (currentLayer) currentLayer.resetStyle(layer);
        document.getElementById('mapOverlay').style.display = 'none';
    }});

    layer.on('click', function() {{
        map.fitBounds(layer.getBounds(), {{ padding: [40, 40] }});
    }});
}}

/* ── RENDER ── */
function renderMap() {{
    if (currentLayer) map.removeLayer(currentLayer);
    currentLayer = L.geoJSON(geojsonKabupaten, {{
        style: styleFeature,
        onEachFeature: onEachFeature
    }}).addTo(map);

    if (currentLayer.getBounds().isValid())
        map.fitBounds(currentLayer.getBounds(), {{ padding: [16, 16] }});

    updateStats();
    updateSliderFill();
}}

/* ── STATS ── */
function updateStats() {{
    let maxV = -Infinity, minV = Infinity;
    let maxN = '', minN = '';
    let sum = 0, count = 0;

    for (const [nama, yearData] of Object.entries(dataKemiskinan)) {{
        const v = yearData[currentTahun];
        if (v == null) continue;
        if (v > maxV) {{ maxV = v; maxN = nama; }}
        if (v < minV) {{ minV = v; minN = nama; }}
        sum += v; count++;
    }}

    if (!count) return;
    document.getElementById('tertinggiWilayah').textContent = displayNama(maxN);
    document.getElementById('tertinggiValue').textContent   = formatRupiah(maxV);
    document.getElementById('terendahWilayah').textContent  = displayNama(minN);
    document.getElementById('terendahValue').textContent    = formatRupiah(minV);
    document.getElementById('rataProvinsi').textContent     = formatRupiah(Math.round(sum / count));
}}

/* ── SLIDER ── */
function updateSliderFill() {{
    const pct = ((parseInt(currentTahun) - 2005) / 20) * 100;
    document.getElementById('yearSlider').style.setProperty('--pct', pct + '%');
}}

function onSliderInput(val) {{
    currentTahun = val;
    document.getElementById('tahunDisplay').textContent = val;
    renderMap();
}}

/* ── PLAYBACK ── */
function doPlay() {{
    if (playTimer) return;
    document.getElementById('playBtn').disabled  = true;
    document.getElementById('pauseBtn').disabled = false;
    playTimer = setInterval(() => {{
        let t = parseInt(currentTahun);
        t = (t >= 2025) ? 2005 : t + 1;
        currentTahun = String(t);
        document.getElementById('yearSlider').value = t;
        document.getElementById('tahunDisplay').textContent = t;
        renderMap();
    }}, 800);
}}

function doPause() {{
    clearInterval(playTimer); playTimer = null;
    document.getElementById('playBtn').disabled  = false;
    document.getElementById('pauseBtn').disabled = true;
}}

function doReset() {{
    doPause();
    currentTahun = '2005';
    document.getElementById('yearSlider').value = 2005;
    document.getElementById('tahunDisplay').textContent = '2005';
    renderMap();
}}

/* ── ZOOM KABUPATEN ── */
function zoomKabupaten(nama) {{
    if (!nama || !currentLayer) return;
    currentLayer.eachLayer(function(layer) {{
        const namaStd = normalisasiNama(getNamaFeature(layer.feature));
        if (namaStd === nama)
            map.fitBounds(layer.getBounds(), {{ padding: [60, 60] }});
    }});
}}

function resetView() {{
    document.getElementById('pilihKabupaten').value = '';
    if (currentLayer && currentLayer.getBounds().isValid())
        map.fitBounds(currentLayer.getBounds(), {{ padding: [16, 16] }});
}}

/* ── INIT ── */
renderMap();
</script>
</body>
</html>"""

In [ ]:
# Simpan Output
output_path = '/content/Peta_Kemiskinan_SumBarrr.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f"  Output : {output_path}")
print(f"  Kabupaten : {len(data_kemiskinan)} wilayah")
print(f"  Tahun     : 2005 – 2025 ({len(TAHUN_LIST)} titik data)")

  Output : /content/Peta_Kemiskinan_SumBar8.html
  Kabupaten : 19 wilayah
  Tahun     : 2005 – 2025 (21 titik data)
